In [1]:
%pip install pyDOE

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [2]:
from google.colab import drive #add the utilities and data folders to google drive
drive.mount('/content/drive') #and mount it on google colab

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/burgers_shock_mu_01_pi.mats') #upload utilities folder to your google drive before running the lines

In [4]:
import tensorflow as tf
import datetime, os
#hide tf logs 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'} 
#0 (default) shows all, 1 to filter out INFO logs, 2 to additionally filter out WARNING logs, and 3 to additionally filter out ERROR logs
import scipy.optimize
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import time
from pyDOE import lhs       #Latin Hypercube Sampling
# import lhs

# specify random seeds for reproducible results
np.random.seed(3)
tf.random.set_seed(3)

In [5]:
data = scipy.io.loadmat('../content/drive/MyDrive/burgers_shock_mu_01_pi.mat')  	# Load data
x = data['x']                                   # 256 points between -1 and 1 [256x1]
t = data['t']                                   # 100 time points between 0 and 1 [100x1] 
usol = data['usol']                             # solution of 256x100 grid points used only for testing (not for training)
X, T = np.meshgrid(x,t) 

# 1. burgers_shock_mu_01_pi.mat == burgers_shock.mat
# - mu = 0.01/pi, IC: -sin(pi*x), BC: u(−1, t) = u(1, t) = 0

# 2. burgers_shock_mu_005_pi.mat
# - mu = 0.005/pi, IC = -sin(pi*x), BC: u(−1, t) = u(1, t) = 0

# 3. burgers_shock_IC_sin2pi.mat
# - mu = 0.01/pi, IC = -sin(2*pi*x), BC: u(−1, t) = u(1, t) = 0

In [6]:
X_u_test = np.hstack((X.flatten()[:,None], T.flatten()[:,None]))

# Domain bounds
lb = X_u_test[0]  # [-1. 0.]
ub = X_u_test[-1] # [1.  0.99]

'''
   Fortran Style ('F') flatten,stacked column wise!
   u = [c1 
        c2
        .
        .
        cn]

   u =  [25600x1] 
'''
u = usol.flatten('F')[:,None] # fortran style was used to make solution points compatible with (x,t) coordinate points

In [7]:
def trainingdata(N_u,N_f): ### randomize the choice on baoundary points

    '''Boundary Conditions'''

    #Initial Condition -1 =< x =<1 and t = 0  
    leftedge_x = np.hstack((X[0,:][:,None], T[0,:][:,None])) #L1
    leftedge_u = usol[:,0][:,None]

    #Boundary Condition x = -1 and 0 =< t =<1
    bottomedge_x = np.hstack((X[:,0][:,None], T[:,0][:,None])) #L2
    bottomedge_u = usol[-1,:][:,None]

    #Boundary Condition x = 1 and 0 =< t =<1
    topedge_x = np.hstack((X[:,-1][:,None], T[:,0][:,None])) #L3
    topedge_u = usol[0,:][:,None]

    all_X_u_train = np.vstack([leftedge_x, bottomedge_x, topedge_x]) # X_u_train [456,2] (456 = 256(L1)+100(L2)+100(L3))
    all_u_train = np.vstack([leftedge_u, bottomedge_u, topedge_u])   #corresponding u [456x1]

    #choose random N_u points for training
    idx = np.random.choice(all_X_u_train.shape[0], N_u, replace=False) 

    X_u_train_rand = all_X_u_train[idx, :] #choose indices from  set 'idx' (x,t) #### randomize boundary point selection like in
    #### collocation points
    u_train_rand = all_u_train[idx,:]      #choose corresponding u

    '''Collocation Points'''

    # Latin Hypercube sampling for collocation points 
    # N_f sets of tuples(x,t)
#     X_f_train = lb + (ub-lb)*pyDOE.lhs(2,N_f)  ### try other collocation point techniques
    X_f_train = lb + (ub-lb)*lhs(2,N_f)
#     print(X_f_train.shape)
    X_f_train = np.vstack((X_f_train, X_u_train_rand)) # append training points to collocation points 

    return X_f_train, X_u_train_rand, u_train_rand 

In [8]:
class PINN(tf.Module): 
    def __init__(self, name=None):
        super().__init__(name = name)
        self.loss_history = np.array([])
        self.nu_history = np.array([])
        self.w_history = np.array([])
        self.nu = tf.Variable(1.2, dtype='float64', trainable = True)
        self.num_inter = 0
    def NN(self, layers):
        ### if you need to get the weights from user, pass it to the
        ### the function. otherwise, do not pass the weights.
        self.layers = layers
        self.W = []
        for i in range(len(layers)): ### enumerate
            if i==len(layers)-1: break
            w = tf.random.normal([layers[i],layers[i+1]], dtype = 'float64')
            w = tf.Variable(w, trainable = True)
            b = tf.zeros([1, layers[i+1]], dtype='float64', name=None) ### no bias for last output
            b = tf.Variable(b, trainable = True)
### edit tf.zeros for flexibility with other layer types
            self.W.append(w)
            self.W.append(b)
#         return W ### is return really necessary?
    
    def forward(self, inpu): ### do not specify input type
#         pass 
### try adding a normalization function 
        self.inpu = inpu
#         print('inpu.shape:', inpu.shape)
        inner = inpu
        for i, layer in enumerate(self.W):
            if i==len(self.W)/2 : break ### half of the length of layers ## lenth of W is always even so we don't care
            ## about the remnant of the devision.
            inner = tf.matmul(inner, self.W[2*i])
            inner = tf.add(inner, self.W[2*i+1]) ### add activation function, now it's unit activation
            if i==len(self.W)/2-1: break # no activation for last layer
            # if i==3: break
#         inner = tf.nn.tanh(inner) # bug: do not specify union function as activation coz the second derivative is zero
        # however, gradient tape returns none type and returns incompatibility type error for the rest of calculations
            # inner = tf.nn.relu(inner)
            inner = tf.math.sigmoid(inner)
        outpu = inner
        return outpu
        
    def PDE(self, X_f_train): 
#         self.nu = tf.Variable(0.003, dtype='float64', trainable = True)
#         print(nu)
#         nu = 0.01/np.pi ### use it in callback and maybe in a tape.
        ### but i'm 70% sure it shouldn't be in the next callback as we try to treat it as a NN weight so scipy optimize
        ### will deal with it.
#         print('X_f_train in PDE:', X_f_train.shape)
        xt_var = tf.Variable(X_f_train, dtype='float64', trainable = False)
#         xt_var = tf.Variable(X_f_train, dtype='float64', trainable = False) ### try other dtypes here and in general
        ### check it it should be trainable
        # we meade a variable coz tape.gradient only takes the derivation of input with respect to a variable
        x_var = xt_var[:,0:1]
        t_var = xt_var[:,1:2]
#         print(x_var)
#         print(t_var)
        with tf.GradientTape(persistent = True) as tape1:
            tape1.watch(x_var)
            tape1.watch(t_var)
            with tf.GradientTape(persistent = True) as tape2:
                tape2.watch(x_var)
                tape2.watch(t_var)
                proxy_xt = tf.stack([x_var[:,0], t_var[:,0]], axis=1)
                u_pred = self.forward(proxy_xt)
            u_pred_x = tape2.gradient(u_pred, x_var) # gradient of u_pred with respect to x_var
        u_pred_t = tape2.gradient(u_pred, t_var) # gradient of u_pred with respect to x_var 
        u_pred_xt = tape1.gradient(u_pred_x, t_var) # gradient of u_pred_x with respect to t_var
        u_pred_xx = tape1.gradient(u_pred_x, x_var) # gradient of u_pred_x with respect to x_var
        # a general role for tape: specify one colomn arrays for avoiding complexity (esp for targets)
#         print('x_var')
#         print(x_var)
#         print('proxy_xt')
#         print(proxy_xt)
#         print('u_pred')
#         print(u_pred)
#         print('u_pred_x')
#         print(u_pred_x)
#         print('u_pred_t')
#         print(u_pred_t)
#         print('u_pred_xx')
#         print(u_pred_xx)
#         print('u_pred_xt')
#         print(u_pred_xt)
#         print(type(u_pred_xx))
        del tape1
        del tape2
        
#         u_pred_again = self.forward(proxy_xt) ### because everything after 'del tape' gets deleted but should we define it
        ### again?
        
        f = u_pred_t + u_pred*u_pred_x - self.nu*u_pred_xx
#         print(u_pred_xx.shape)
        
        return f

    def PPDE(self, X_f_train, X_u_train, u_train): ### it's the same as PDE except that the source of gradient has 2 column
        xt_var = tf.Variable(X_f_train, dtype='float64', trainable = False)
        ### try other dtypes here and in general
        # we made a variable coz tape.gradient only takes the derivation of input with respect to a variable
#         x_var = xt_var[:,0:1]
#         t_var = xt_var[:,1:2]
        with tf.GradientTape(persistent = True) as tape1:
            tape1.watch(xt_var)
            with tf.GradientTape(persistent = True) as tape2:
                tape2.watch(xt_var)
#                 tape2.watch(x_var)
#                 tape2.watch(t_var)
#                 proxy_xt = tf.stack([x_var[:,0], t_var[:,0]], axis=1)
                u_pred = self.forward(xt_var)
            u_pred_x = tape2.gradient(u_pred, xt_var)[:,0:1] # gradient of u_pred with respect to x_var
            u_pred_t = tape2.gradient(u_pred, xt_var)[:,1:2] # gradient of u_pred with respect to t_var
        u_pred_xx_xt = tape1.gradient(u_pred_x, xt_var) # gradient of u_pred_x with respect to both x and t
        u_pred_xx = u_pred_xx_xt[:,0:1]
        u_pred_xt = u_pred_xx_xt[:,1:2]
#         u_pred_t = tape.gradient(u_pred, t_var) # gradient of u_pred with respect to x_var
        del tape1
        del tape2

#         u_pred_again = self.forward(proxy_xt) ### because everything after 'del tape' gets deleted but should we define it
#         ### again?

        f = u_pred_t + u_pred*u_pred_x - nu*u_pred_xx

        return f

    def loss(self, X_f_train, X_u_train, u_train):
        X_u_train = tf.Variable(X_u_train, dtype='float64', trainable = False)
        u_train = tf.Variable(u_train, dtype='float64', trainable = False)
        e_DD = self.forward(X_u_train) - u_train
        loss_DD = tf.reduce_mean(tf.square(e_DD))
        
        f = pinn.PDE(X_f_train)
        loss_PI = tf.reduce_mean(tf.square(f)) # physics-informed loss
        
        loss = 0.5 * loss_PI + 0.5 * loss_DD ### try different weights
        ### add an extra term in the loss for training loss in the collocation. it should be exactly the same as boundary.
        return loss, loss_DD, loss_PI
    
    def set_weights(self,parameters):
                
        for i in range (len(self.layers)-1):

            shape_w = tf.shape(self.W[2*i]).numpy() # shape of the weight tensor
            size_w = tf.size(self.W[2*i]).numpy() #size of the weight tensor 
            
            shape_b = tf.shape(self.W[2*i+1]).numpy() # shape of the bias tensor
            size_b = tf.size(self.W[2*i+1]).numpy() #size of the bias tensor 
                        
            pick_w = parameters[0:size_w] #pick the weights 
            self.W[2*i].assign(tf.reshape(pick_w,shape_w)) # assign  
            parameters = np.delete(parameters,np.arange(size_w),0) #delete 
            
            pick_b = parameters[0:size_b] #pick the biases 
            self.W[2*i+1].assign(tf.reshape(pick_b,shape_b)) # assign 
            parameters = np.delete(parameters,np.arange(size_b),0) #delete
        
        self.nu.assign(parameters[-1]) # the last parameter is the pde parameter. try adding several parameters
        
    def get_weights(self):

        parameters_1d = []  # [.... W_i,b_i.....  ] 1d array
        
        for i in range (len(self.layers)-1):
            
            w_1d = tf.reshape(self.W[2*i],[-1])   #flatten weights 
            b_1d = tf.reshape(self.W[2*i+1],[-1]) #flatten biases
            
            parameters_1d = tf.concat([parameters_1d, w_1d], 0) #concat weights 
            parameters_1d = tf.concat([parameters_1d, b_1d], 0) #concat biases
            
        nu_1d = tf.reshape(self.nu,[-1])    #flatten pde params 
        parameters_1d = tf.concat([parameters_1d, nu_1d], 0) #concat pde params
        return parameters_1d    
        
    
    def backward(self,parameters):
        self.set_weights(parameters) # sets weights as a matrix

        with tf.GradientTape() as tape:
            tape.watch(self.trainable_variables)

            loss,_,_ = self.loss(X_f_train, X_u_train, u_train) ### can we solve the problem with just loss_DD or loss_PI?

        grads = tape.gradient(loss, self.trainable_variables) ### includes nu, which is the last parameter

        del tape

        grads_1d = [ ] #flatten grads 
        num_param = 1
        
        # so far the grads are in matrix form, by the following loop we'll change it to 1 dimentional array. 
        ### because scipy.optimize speaks in the language of one dimnetional arrays
        for i in range(len(self.layers)-1):

            grads_w_1d = tf.reshape(grads[2*i],[-1]) #flatten weights 
            grads_b_1d = tf.reshape(grads[2*i+1],[-1]) #flatten biases

            grads_1d = tf.concat([grads_1d, grads_w_1d], 0) #concat grad_weights 
            grads_1d = tf.concat([grads_1d, grads_b_1d], 0) #concat grad_biases
            
        grads_nu_1d = tf.reshape(grads[-1],[-1])
        grads_1d = tf.concat([grads_1d, grads_nu_1d], 0)
        return loss.numpy(), grads_1d.numpy()
    
    def callback(self,parameters):
        self.num_inter += 1
               
        loss,_,_ = self.loss(X_f_train, X_u_train, u_train)
        loss_np = loss.numpy()
        nu_np = self.nu.numpy()
        print('num_inter:', self.num_inter, '\t', 'loss_history_np:', loss_np, '\t', 'nu_np:', nu_np)
#         print('nu_np',nu_np)
        
        self.loss_history = np.append(self.loss_history, loss_np)
        self.nu_history = np.append(self.nu_history, nu_np)
        self.w_history = np.append(self.w_history, self.W[0][0][0]) ### putting the probe on a single weight amongst all.
        
        

        
## add the pde paramter to PDE()       

In [9]:
pinn = PINN(name='first')
nn = pinn.NN(layers = [2, 20, 20, 20, 20, 1]) # initializing nn weights and biases # number of layers N=5
### it's best to try initializing the PDE param as well.
# for i in range(len(pinn.W)):
#     print(pinn.W[i].shape)


In [10]:
N_u = 100
N_f = 10000
X_f_train, X_u_train, u_train = trainingdata(N_u,N_f) 
# f = pinn.PDE(X_f_train) # boundary and collocations points (N_u + N_f) give an array of PDE residuals
# f = pinn.PPDE(X_f_train, X_u_train, u_train)
# loss,_,_ = pinn.loss(X_f_train, X_u_train, u_train)
init_params = pinn.get_weights().numpy()
# print(X_f_train.shape)

In [11]:
t_start = time.time()
results = scipy.optimize.minimize(fun = pinn.backward, 
                                  x0 = init_params, 
                                  args=(), 
                                  method='L-BFGS-B', 
                                  jac= True,
                                  callback = pinn.callback,
                                  options = {'disp': None,
                                            'maxcor': 200, 
                                            'ftol': 1 * np.finfo(float).eps,  #The iteration stops when (f^k - f^{k+1})/max{|f^k|,|f^{k+1}|,1} <= ftol
                                            'gtol': 5e-8, 
                                            'maxfun':  50000, 
                                            'maxiter': 5000,
                                            'iprint': -1,   #print update every 50 iterations
                                            'maxls': 50})
run_time = time.time() - t_start 
print('run_time:', run_time)

ValueError: ignored

In [ ]:
fig, ax = plt.subplots()  
ax.plot(pinn.loss_history)
# ax.set_xlim(5, 0)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('loss', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('evolution of loss\n', fontsize = 14, fontweight ='bold')
plt.savefig('loss.png')
plt.show()

In [ ]:
fig, ax = plt.subplots()  
ax.plot(pinn.nu_history)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('pde parameter nu', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('evolution of lambda\n', fontsize = 14, fontweight ='bold')
plt.savefig('nu.png')
plt.show()

In [ ]:
pinn.w_history

In [ ]:
fig, ax = plt.subplots()  
ax.plot(pinn.w_history)
ax.set_xlabel('iterations', 
               fontweight ='bold')
ax.set_ylabel('w_history', 
               fontweight ='bold')
# ax.grid(True)
  
ax.set_title('training history of the first weight in the NN\n', fontsize = 14, fontweight ='bold')
plt.savefig('w0.png')
plt.show()

In [ ]:
from google.colab import files
files.download("loss.png")
files.download("nu.png")
files.download("w0.png")